Import all necessary libraries, covering parsing logs,labeling, analysis, recons

In [1]:
import pandas as pd
from pathlib import Path

# -------- parsing ----------
from parsing.syslog import parse_syslog_csv
from parsing.auditd import parse_audit_log
from parsing.auth import parse_auth_log
from parsing.zeek import (
    parse_conn_log, parse_dns_log, parse_http_log, 
    parse_files_log, parse_ssl_log, parse_ssh_log
)
from parsing.suricata import parse_suricata_eve
from parsing.azure_wins import (
    parse_azure_conn, parse_azure_process, parse_azure_security,
    parse_azure_events, parse_azure_port, parse_azure_perf
)
from parsing.tracee import parse_tracee_ndjson

# ---------- labeling / analysis / recons ----------
from labeling.tagger import tag_steps
from analysis.coverage import step_coverage
from analysis.failure_taxonomy import failure_taxonomy 
from analysis.metrics import compute_metrics
from analysis.ambig import ambiguity

from recons.event_graph import build_event_graph
from recons.chain_builder import reconstruct_chain_detailed
# --- scenarios config ---
from scenarios import SCENARIOS



import json

# expected-step scoring from ATT&CK technique layers
try:
    from labeling.rules import expected_steps_from_techniques
except Exception:
    from rules import expected_steps_from_techniques


define the mapping from source -> parser

In [2]:
from typing import Callable, List, Tuple, Dict

def _find_files(log_root: Path, patterns: List[str]) -> List[Path]:
    ''' recursively find files matching any of the given patterns in log_root

    '''
    hits = []
    for pat in patterns:
        hits.extend(log_root.rglob(pat) if not pat.startswith("**/") else log_root.rglob(pat[3:]))
    # remove duplicates while preserving order
    uniq = sorted(set([p for p in hits if p.is_file()]))
    return uniq


def _parse_many(files: List[Path], parser: Callable[[str], pd.DataFrame], *, source: str, source_kind: str) -> pd.DataFrame:
    ''' Multiple files are parsed and merged using the same parser, and source fields are added.
    
    '''
    frames = []
    for p in files:
        try:
            df = parser(str(p))
        except Exception as e:
            print(f"Warning: failed to parse {p} with {parser.__name__}")
            print("The reason was:", e)
            continue
        if df is None or len(df) == 0:
            continue
        df = df.copy()
        df["source"] = source
        df["source_kind"] = source_kind
        df["source_file"] = str(p)
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def _parse_prefixed_components(
    log_root: Path,
    *,
    prefix: str, # "zeek"/"azure"
    component_parsers: Dict[str, Callable[[str], pd.DataFrame]],
    source: str,
) -> pd.DataFrame:
    '''
    recursively find and parse log files for components with given prefix
    '''
    frames = []
    items = component_parsers.items() if isinstance(component_parsers, dict) else component_parsers

    for comp, fn in items:
        patterns = [
            f"{prefix}_{comp}.csv",
            f"{prefix}_{comp}_*.csv",
            f"{prefix}_{comp}*.csv",
        ]
        files = _find_files(log_root, patterns)
        df = _parse_many(files, fn, source=source, source_kind=f"{source}_{comp}")
        if len(df):
            frames.append(df)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def parse_source(source_key: str, log_root: Path) -> pd.DataFrame:
    '''
    return unified source dataframe for a given source_key
    args:
        log_root: Path to scenario log root
    '''
    # --- syslog ---
    if source_key == "syslog":
        files = _find_files(log_root, ["azure_syslog.csv", "azure_syslog*.csv"])
        return _parse_many(files, parse_syslog_csv, source="syslog", source_kind="syslog")
    
    # --- auditd ---
    if source_key == "auditd":
        files = _find_files(log_root, ["audit.log", "auditd*.log"])
        return _parse_many(files, parse_audit_log, source="auditd", source_kind="auditd")

    # --- auth ---
    if source_key == "auth":
        files = _find_files(log_root, ["auth.log", "auth*.log"])
        return _parse_many(files, parse_auth_log, source="auth", source_kind="auth")

    # ----- suricate -----
    if source_key == "suricata":
        files = _find_files(log_root, ["eve.json", "eve*.json"])
        return _parse_many(files, parse_suricata_eve, source="suricata", source_kind="suricata")

    # ----- tracee -------
    if source_key == "tracee":
        files = _find_files(log_root, ["tracee-events.json"])
        return _parse_many(files, parse_suricata_eve, source="tracee", source_kind="tracee")


    # --- zeek (multi-log) ---
    if source_key == "zeek":
        zeek_parsers: List[Tuple[str, Callable[[str], pd.DataFrame]]] = [
            ("conn",  parse_conn_log),
            ("dns",   parse_dns_log),
            ("http",  parse_http_log),
            ("files", parse_files_log),
            ("ssh",   parse_ssh_log),
            ("ssl",   parse_ssl_log),
        ]
        return _parse_prefixed_components(
            log_root,
            prefix="zeek",
            component_parsers=zeek_parsers,
            source="zeek",
        )

    # --- azure windows ---
    if source_key.startswith("azure_"):
        comp = source_key.replace("azure_", "", 1)  # conn/process/security/events/port/syslog...
        azure_component_parsers = {
            "conn":     parse_azure_conn,
            "process":  parse_azure_process,
            "security": parse_azure_security,
            "events":   parse_azure_events,
            "port":     parse_azure_port,     
            "syslog":   parse_syslog_csv,   
        }
        fn = azure_component_parsers.get(comp)
        if fn is None:
            return pd.DataFrame()

        patterns = [
            f"azure_{comp}.csv",
            f"azure_{comp}_*.csv",
            f"azure_{comp}*.csv",
        ]
        files = _find_files(log_root, patterns)
        return _parse_many(files, fn, source=source_key, source_kind=source_key)

    return pd.DataFrame()


run pipeline (tag + metrics + chain) with given sources

In [3]:
def _normalize_ts(df: pd.DataFrame, ts_col="ts") -> pd.DataFrame:
    if df is None or len(df) == 0 or ts_col not in df.columns:
        return df
    out = df.copy()

    ts = pd.to_datetime(out[ts_col], errors="coerce", utc=True)

    out[ts_col] = ts.dt.tz_convert(None)
    return out


def run_pipeline_for_sources(enabled_sources, log_root: Path, expected_steps=None):
    """Run parse->tag->analyze->reconstruct for a given enabled source set.

    expected_steps:
      Optional list of coarse steps (e.g., ["AUTH","DOWNLOAD"]) derived from ATT&CK techniques.
      If provided, reconstruction quality is scored as step-level recall/precision against this set.
    """
    # 1) parsing
    frames = []

    for s in enabled_sources:
        df = parse_source(s, log_root)
        if df is not None and len(df) > 0:
            df["source"] = s
            frames.append(df)

    events = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    # --- guard: prevent cross-scenario leakage ---
    if len(events) and "source_file" in events.columns:
        root = str(Path(log_root).resolve())
        events = events[events["source_file"].astype(str).str.startswith(root)].copy()

    # 2) tagging (with schema diagnostics)
    if len(events):
        tagged, tag_stats = tag_steps(events, return_stats=True)
    else:
        tagged, tag_stats = events, {}

    if len(tagged) and "ts" in tagged.columns:
        tagged = tagged.copy()
        tagged["ts"] = pd.to_datetime(tagged["ts"], utc=True, errors="coerce").dt.tz_convert(None)

    # 3) analytics
    cov = step_coverage(tagged) if len(tagged) else {}
    fails = failure_taxonomy(tagged) if len(tagged) else {}
    amb = ambiguity(tagged) if len(tagged) else {}

    # 4) reconstruction
    g = None
    chain_steps = []
    if len(tagged) and "step" in tagged.columns and "ts" in tagged.columns:
        g = build_event_graph(tagged)
        d = reconstruct_chain_detailed(g, tagged.dropna(subset=["step"]))
        chain_steps = d.get("steps", [])

    # 5) metrics (chain-aware)
    metrics = compute_metrics(tagged, chain=chain_steps, max_gap_seconds=600) if len(tagged) else {}

    # 6) reconstruction summary
    observed_steps = []
    if len(tagged) and "step" in tagged.columns:
        observed_steps = sorted(tagged["step"].dropna().unique().tolist())

    expected_steps = expected_steps or []
    expected_set = set(expected_steps)
    observed_set = set(observed_steps)
    chain_set = set(chain_steps)

    step_recall = (len(observed_set & expected_set) / len(expected_set)) if expected_set else None
    step_precision = (len(observed_set & expected_set) / len(observed_set)) if observed_set else None

    chain_recall = (len(chain_set & expected_set) / len(expected_set)) if expected_set else None
    chain_precision = (len(chain_set & expected_set) / len(chain_set)) if chain_set else None

    # Backward-compatible reconstructability:
    # - prefer expected-step recall when available
    # - otherwise fall back to old ratio (chain/observed)
    if expected_set:
        reconstructability = step_recall
    else:
        reconstructability = (len(chain_steps) / len(observed_steps)) if observed_steps else 0.0

    # compact tagger diagnostics (helps explain silent misses)
    diag = {
        "tagger_total_hits": 0,
        "tagger_missing_fields_events": 0,
        "tagger_missing_prefilter_events": 0,
        "tagger_used_prefilter_conditions": 0,
        "tagger_unusable_rules": [],
    }
    if isinstance(tag_stats, dict) and "rules" in tag_stats:
        for rid, rs in tag_stats["rules"].items():
            diag["tagger_total_hits"] += int(rs.get("hits", 0) or 0)
            diag["tagger_missing_fields_events"] += int(rs.get("missing_fields_events", 0) or 0)
            diag["tagger_missing_prefilter_events"] += int(rs.get("missing_prefilter_events", 0) or 0)
            diag["tagger_used_prefilter_conditions"] += int(rs.get("used_prefilter_conditions", 0) or 0)
            if (rs.get("hits", 0) or 0) == 0 and (rs.get("missing_fields_events", 0) or 0) > 0:
                diag["tagger_unusable_rules"].append(rid)

    recon = {
        "n_events": int(len(events)) if len(events) else 0,
        "expected_steps": sorted(expected_set),
        "n_expected_steps": int(len(expected_set)),
        "observed_steps": observed_steps,
        "n_tagged_steps": int(len(observed_steps)),
        "chain_steps": chain_steps,
        "n_chain_steps": int(len(chain_steps)),
        "reconstructability": reconstructability,
        "step_recall": step_recall,
        "step_precision": step_precision,
        "chain_recall": chain_recall,
        "chain_precision": chain_precision,
        **diag,
    }

    return {
        "events": events,
        "tagged": tagged,
        "coverage": cov,
        "failures": fails,
        "ambiguity": amb,
        "metrics": metrics,
        "chain": chain_steps,
        "graph": g,
        "recon": recon,
    }


### comparison between single source vs multiple sources

- multiple sources: all sources from one scenario
- single source: every source in one scenarios separately runs once, find failure mode and missing part
- output one summary

In [4]:
def get_expected_sources_for_scenario(sc) -> list:
    # sc.expected_sources is dict[str,bool]
    return [k for k, v in sc.expected_sources.items() if v]


def _load_layer_techniques(layer_path: Path) -> list:
    """Load techniqueIDs from an ATT&CK Navigator layer JSON."""
    try:
        j = json.loads(layer_path.read_text())
    except Exception:
        return []

    techs = []
    for t in j.get("techniques", []) or []:
        tid = t.get("techniqueID")
        if tid:
            techs.append(tid)
    return techs


def load_scenario_ttps(log_root: Path) -> list:
    """Best-effort discovery of ATT&CK layer files for a scenario.

    Order:
      1) scenario-local *ttps*.json / attack_navigator_*.json
      2) fallback to repository-level defaults if present
    """
    log_root = Path(log_root)

    layer_files = []
    layer_files += sorted(log_root.glob("*ttp*.json"))
    layer_files += sorted(log_root.glob("*ttps*.json"))
    layer_files += sorted(log_root.glob("attack_navigator_*.json"))

    # fallback: common filenames in repo root
    for fn in ["payload_ttps_0.json", "attack_navigator_tasks.json", "attack_navigator_cmds.json"]:
        p = Path(fn)
        if p.exists():
            layer_files.append(p)

    techs = []
    for f in layer_files:
        techs.extend(_load_layer_techniques(f))

    # de-dup preserve sort
    return sorted(set(techs))


def analyze_scenario(scenario_key: str, log_root: Path):
    log_root = Path(log_root)

    # if caller passed the global root, resolve scenario dir automatically
    cands = [
        log_root / scenario_key,
        log_root / scenario_key.lower(),
        log_root / scenario_key.upper(),
    ]
    for d in cands:
        if d.exists():
            log_root = d
            break

    # extract scenario config
    sc = SCENARIOS[scenario_key]
    expected_sources = get_expected_sources_for_scenario(sc)

    # expected steps derived from ATT&CK layer(s) (if available)
    technique_ids = load_scenario_ttps(log_root)
    expected_steps = expected_steps_from_techniques(technique_ids)

    # ---- multi-source run -----
    multi = run_pipeline_for_sources(expected_sources, log_root, expected_steps=expected_steps)

    # --- single-source runs ---
    single_rows = []
    single_outputs = {}
    for s in expected_sources:
        out = run_pipeline_for_sources([s], log_root, expected_steps=expected_steps)
        single_outputs[s] = out
        single_rows.append({
            "scenario": sc.id,
            "name": sc.name,
            "mode": "single",
            "source_set": s,
            **out["recon"],
            "failure_keys": sorted(list(out["failures"].keys())) if isinstance(out["failures"], dict) else [],
        })

    # best single-source by recon
    single_df = pd.DataFrame(single_rows)
    if len(single_df):
        best_single = single_df.sort_values(
            ["reconstructability", "n_chain_steps", "n_tagged_steps"],
            ascending=False,
        ).head(1)
        best_single_row = best_single.iloc[0].to_dict()
    else:
        best_single_row = {}

    # multi row
    multi_row = {
        "scenario": sc.id,
        "name": sc.name,
        "mode": "multi",
        "source_set": "+".join(expected_sources),
        **multi["recon"],
        "failure_keys": sorted(list(multi["failures"].keys())) if isinstance(multi["failures"], dict) else [],
    }

    # --- Gap explanation helpers (for Q2/Q3) ---
    # Prefer expected steps (from ATT&CK) for gap explanations. If unavailable, fall back to steps observed in multi.
    expected_step_set = set(expected_steps or [])
    multi_steps = set(multi["recon"].get("observed_steps", []))

    step_presence = {}
    for s, out in single_outputs.items():
        step_presence[s] = set(out["recon"].get("observed_steps", []))

    basis_steps = sorted(expected_step_set) if expected_step_set else sorted(multi_steps)

    step_missing_in_single = {}
    for st in basis_steps:
        missing_sources = [s for s in expected_sources if st not in step_presence.get(s, set())]
        if missing_sources:
            step_missing_in_single[st] = missing_sources

    expected_steps_missing_in_multi = sorted(list(expected_step_set - multi_steps)) if expected_step_set else []

    # Which failures are most frequent in single-source scenarios but mitigated by multi-source scenarios?
    def failure_keys(x):
        return set(x.keys()) if isinstance(x, dict) else set()

    multi_fail = failure_keys(multi["failures"])
    single_fail_union = set()
    for s in expected_sources:
        single_fail_union |= failure_keys(single_outputs[s]["failures"])

    mitigated_failures = sorted(list(single_fail_union - multi_fail))

    summary = {
        "scenario": sc.id,
        "name": sc.name,
        "expected_sources": expected_sources,
        "expected_techniques": technique_ids,
        "expected_steps": expected_steps,
        "expected_steps_missing_in_multi": expected_steps_missing_in_multi,
        "multi_reconstructability": multi["recon"]["reconstructability"],
        "best_single_source": best_single_row.get("source_set"),
        "best_single_reconstructability": best_single_row.get("reconstructability", 0.0),
        "multi_minus_best_single": multi["recon"]["reconstructability"] - best_single_row.get("reconstructability", 0.0),
        "steps_missing_in_single": step_missing_in_single,
        "failures_mitigated_by_multi": mitigated_failures,

        # keep old keys for compatibility
        "expected_payload_ttps": technique_ids,
        "expected_c2_ttps": None,
    }

    return {
        "multi": multi,
        "single": single_outputs,
        "single_df": single_df,
        "multi_row": multi_row,
        "summary": summary,
    }


In [5]:
def pretty_three_questions(summary: dict) -> str:
    sc = summary["scenario"]
    name = summary["name"]
    multi_r = summary["multi_reconstructability"]
    best_s = summary["best_single_source"]
    best_r = summary["best_single_reconstructability"]
    delta = summary["multi_minus_best_single"]

    # Q1: end-to-end reconstructability
    q1 = (
        f"[Q1] {sc} ({name}) — End-to-end reconstructability (expected-step recall when available) under multi-source telemetry: "
        f"reconstructability={multi_r:.2f}. "
        f"Best single-source baseline ({best_s})={best_r:.2f} "
        f"(multi − best_single = {delta:+.2f})."
    )

    # Q2: where/why single-source fails (proxy via steps missing in single-source runs)
    missing = summary["steps_missing_in_single"]
    top_missing = list(missing.items())[:8]  # keep it readable

    if top_missing:
        q2_lines = [
            f"  - step={st} is absent in single-source runs for: {srcs}"
            for st, srcs in top_missing
        ]
        q2 = (
            "[Q2] Single-source failure points — examples of steps observable in the multi-source run "
            "but missing under individual sources:\n"
            + "\n".join(q2_lines)
        )
    else:
        q2 = (
            "[Q2] Single-source failure points — no clear step-level omissions detected "
            "(this can happen when the chain is short, when sources have overlapping visibility, "
            "or when labeling evidence is sparse)."
        )

    # Q3: how multi-source mitigates failures (proxy via failure categories that disappear in multi-source)
    mitigated = summary["failures_mitigated_by_multi"]

    if mitigated:
        q3 = (
            "[Q3] Multi-source mitigation — failure categories observed in at least one single-source run "
            "but not present in the multi-source run:\n"
            + "\n".join([f"  - {x}" for x in mitigated[:10]])
        )
    else:
        q3 = (
            "[Q3] Multi-source mitigation — no clear difference in failure keys "
            "(consider refining this with step-level failure counts and/or explicit observability-gap signals)."
        )

    return q1 + "\n" + q2 + "\n" + q3


In [6]:
LOG_ROOT = Path.cwd().parent.joinpath("sanidata")

print("LOG_ROOT:", LOG_ROOT.resolve())
print("LOG_ROOT exists:", LOG_ROOT.exists())

all_files = [p for p in LOG_ROOT.rglob("*") if p.is_file()]
print("n_files under LOG_ROOT:", len(all_files))
print("sample:", [str(p.relative_to(LOG_ROOT)) for p in all_files[:50]])


LOG_ROOT: /Users/zhuoran/Projects/SSCMDataset/sanidata
LOG_ROOT exists: True
n_files under LOG_ROOT: 121
sample: ['.DS_Store', 'sc1/attack_navigator_cmds.json', 'sc1/attack_navigator_tasks.json', 'sc1/payload_ttps_2.json', 'sc1/payload_ttps_3.json', 'sc1/.DS_Store', 'sc1/payload_ttps_0.json', 'sc1/payload_ttps_1.json', 'sc1/windows/azure_port.csv', 'sc1/windows/azure_events.parquet', 'sc1/windows/azure_process.parquet', 'sc1/windows/azure_port.parquet', 'sc1/windows/azure_conn.parquet', 'sc1/windows/azure_process.csv', 'sc1/windows/azure_security.csv', 'sc1/windows/azure_conn.csv', 'sc1/windows/azure_perf.parquet', 'sc1/windows/azure_security.parquet', 'sc1/windows/azure_perf.csv', 'sc1/windows/azure_events.csv', 'sc6/attack_navigator_cmds.json', 'sc6/attack_navigator_tasks.json', 'sc6/payload_ttps_2.json', 'sc6/payload_ttps_3.json', 'sc6/.DS_Store', 'sc6/payload_ttps_4.json', 'sc6/payload_ttps_0.json', 'sc6/payload_ttps_1.json', 'sc6/victim/azure_events.parquet', 'sc6/victim/custom_au

In [7]:

sc1_dir = LOG_ROOT / "sc1"

print("files conn:", list(sc1_dir.rglob("azure_conn.csv")))
print("files conn windows:", list(sc1_dir.rglob("windows/azure_conn.csv")))

df = parse_source("azure_conn", sc1_dir)
print("parsed rows:", len(df))
print("cols:", df.columns.tolist()[:20] if len(df) else None)

files conn: [PosixPath('/Users/zhuoran/Projects/SSCMDataset/sanidata/sc1/windows/azure_conn.csv')]
files conn windows: [PosixPath('/Users/zhuoran/Projects/SSCMDataset/sanidata/sc1/windows/azure_conn.csv')]
parsed rows: 5746
cols: ['ts', 'host', 'source', 'event_type', 'action', 'subject', 'object', 'src', 'dst', 'raw', 'source_kind', 'source_file']


In [8]:
# ---- run (scenario-local root) ----
scenario_keys = list(SCENARIOS.keys())
all_rows = []
all_summaries = {}

for k in scenario_keys:
    res = analyze_scenario(k, LOG_ROOT)  
    all_rows.append(res["multi_row"])
    if len(res["single_df"]):
        all_rows.extend(res["single_df"].to_dict("records"))
    all_summaries[k] = res["summary"]

result_df = pd.DataFrame(all_rows)
result_df.sort_values(["scenario", "mode", "reconstructability"], ascending=[True, True, False], inplace=True)
result_df


/Users/zhuoran/Projects/SSCMDataset/ana/labeling/tagger.py:70: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hit = rule_mask & blob.str.contains(rx, na=False)
/Users/zhuoran/Projects/SSCMDataset/ana/labeling/tagger.py:70: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hit = rule_mask & blob.str.contains(rx, na=False)
/Users/zhuoran/Projects/SSCMDataset/ana/labeling/tagger.py:70: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hit = rule_mask & blob.str.contains(rx, na=False)
/Users/zhuoran/Projects/SSCMDataset/ana/labeling/tagger.py:70: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hit = rule_mask & blob.str.contains(rx, na=False)
/Users/z

,scenario,name,mode,source_set,n_events,n_tagged_steps,n_chain_steps,reconstructability,observed_steps,chain_steps,failure_keys
0,SC1,Stegano,multi,azure_conn+azure_process+azure_security+azure_...,8534,1,1,1.0,[OUTBOUND_CONN],[OUTBOUND_CONN],[]
1,SC1,Stegano,single,azure_conn,5746,1,1,1.0,[OUTBOUND_CONN],[OUTBOUND_CONN],[]
2,SC1,Stegano,single,azure_process,345,0,0,0.0,[],[],[]
3,SC1,Stegano,single,azure_security,334,0,0,0.0,[],[],[]
4,SC1,Stegano,single,azure_events,1000,0,0,0.0,[],[],[]
5,SC1,Stegano,single,azure_port,1109,0,0,0.0,[],[],[]
6,SC2,Starter,multi,azure_events+syslog,53978,0,0,0.0,[],[],[]
7,SC2,Starter,single,azure_events,44246,0,0,0.0,[],[],[]
8,SC2,Starter,single,syslog,9732,0,0,0.0,[],[],[]
9,SC3,Parallel,multi,auditd+auth+suricata+syslog+zeek,88674,2,2,1.0,"[AUTH, OUTBOUND_CONN]","[AUTH, OUTBOUND_CONN]",[]


In [9]:
result_df.to_csv("total_scenarios.csv")